# ViT Training Notebook

In [ ]:
import csv
import json
import random
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import models, transforms

try:
    import pandas as pd
except ImportError:
    pd = None


PROJECT_ROOT = next(
    (
        candidate
        for candidate in [Path.cwd(), *Path.cwd().parents]
        if (candidate / 'data').exists() and (candidate / 'Model').exists()
    ),
    Path.cwd(),
)

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def seed_everything(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def parse_age_from_name(path: Path) -> int:
    stem = path.stem
    if '_' not in stem:
        raise ValueError(f"Filename does not contain '_' age separator: {path.name}")
    age_str = stem.rsplit('_', 1)[1]
    if not age_str.isdigit():
        raise ValueError(f"Age is not numeric in filename: {path.name}")
    return int(age_str)


def build_samples(root: Path) -> List[Tuple[Path, int]]:
    samples = []
    for path in root.rglob('*'):
        if path.is_file() and path.suffix.lower() in IMG_EXTS:
            age = parse_age_from_name(path)
            samples.append((path, age))
    if not samples:
        raise RuntimeError(f'No images found under {root}')
    return samples


def split_indices_by_label(samples, val_ratio: float, seed: int, adult_age: int = 21):
    rng = random.Random(seed)
    by_label = defaultdict(list)
    for idx, (_, age) in enumerate(samples):
        label = 1 if age >= adult_age else 0
        by_label[label].append(idx)

    train_idx, val_idx = [], []
    for indices in by_label.values():
        rng.shuffle(indices)
        n_items = len(indices)
        n_val = int(round(n_items * val_ratio))
        if n_items >= 2:
            n_val = max(1, min(n_items - 1, n_val))
        else:
            n_val = 0
        val_idx.extend(indices[:n_val])
        train_idx.extend(indices[n_val:])

    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    return train_idx, val_idx


class FaceAdultDataset(Dataset):
    def __init__(self, samples, tfm, adult_age: int = 21):
        self.samples = samples
        self.tfm = tfm
        self.adult_age = adult_age

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index: int):
        path, age = self.samples[index]
        image = Image.open(path).convert('RGB')
        image = self.tfm(image)
        label = 1.0 if age >= self.adult_age else 0.0
        return image, torch.tensor(label, dtype=torch.float32)


class ViTBinary(nn.Module):
    def __init__(self, pretrained: bool = False):
        super().__init__()
        weights = models.ViT_B_16_Weights.IMAGENET1K_V1 if pretrained else None
        vit = models.vit_b_16(weights=weights)
        in_features = vit.heads.head.in_features
        vit.heads = nn.Identity()
        self.backbone = vit
        self.head = nn.Linear(in_features, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        return self.head(features).squeeze(1)


@torch.no_grad()
def evaluate(model, loader, device, criterion, threshold: float = 0.5) -> Dict[str, float]:
    model.eval()
    total_loss, total = 0.0, 0
    tp = fp = tn = fn = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)
        probs = torch.sigmoid(logits)
        preds = (probs >= threshold).float()

        batch_size = images.size(0)
        total += batch_size
        total_loss += loss.item() * batch_size

        tp += ((preds == 1) & (labels == 1)).sum().item()
        tn += ((preds == 0) & (labels == 0)).sum().item()
        fp += ((preds == 1) & (labels == 0)).sum().item()
        fn += ((preds == 0) & (labels == 1)).sum().item()

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = (2 * precision * recall) / max(precision + recall, 1e-12)
    acc = (tp + tn) / max(total, 1)

    return {
        'loss': total_loss / max(total, 1),
        'acc': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'tn': tn,
        'fp': fp,
        'fn': fn,
    }


def train_one_epoch(model, loader, optimizer, device, criterion) -> float:
    model.train()
    total_loss, total = 0.0, 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        total += batch_size
        total_loss += loss.item() * batch_size

    return total_loss / max(total, 1)


def write_history(history: List[Dict], csv_path: Path, json_path: Path) -> None:
    if not history:
        return

    fieldnames = list(history[0].keys())
    with csv_path.open('w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(history)

    with json_path.open('w', encoding='utf-8') as file:
        json.dump(history, file, indent=2)


In [ ]:
CONFIG = {
    'data_dir': str(PROJECT_ROOT / 'data' / 'Face_Age_Dataset'),
    'out_dir': str(PROJECT_ROOT / 'Model' / 'ViT' / 'runs' / 'vit_adult_binary'),
    'epochs': 20,
    'batch_size': 16,
    'lr': 1e-4,
    'weight_decay': 1e-4,
    'val_ratio': 0.2,
    'img_size': 224,
    'num_workers': 4,
    'seed': 42,
    'pretrained': False,
    'adult_age': 21,
    'threshold': 0.5,
    'save_every_epoch_checkpoint': False,
}

CONFIG


In [ ]:
seed_everything(CONFIG['seed'])

data_root = Path(CONFIG['data_dir'])
out_dir = Path(CONFIG['out_dir'])
out_dir.mkdir(parents=True, exist_ok=True)

history_csv_path = out_dir / 'history.csv'
history_json_path = out_dir / 'history.json'
best_ckpt_path = out_dir / 'best_model.pt'
last_ckpt_path = out_dir / 'last_model.pt'
config_path = out_dir / 'config.json'
epoch_ckpt_dir = out_dir / 'epoch_checkpoints'
if CONFIG['save_every_epoch_checkpoint']:
    epoch_ckpt_dir.mkdir(parents=True, exist_ok=True)

samples = build_samples(data_root)
adult_count = sum(1 for _, age in samples if age >= CONFIG['adult_age'])
nonadult_count = len(samples) - adult_count

if adult_count == 0 or nonadult_count == 0:
    raise RuntimeError('Dataset must contain BOTH adult and not-adult samples.')

train_idx, val_idx = split_indices_by_label(
    samples,
    CONFIG['val_ratio'],
    CONFIG['seed'],
    adult_age=CONFIG['adult_age'],
)

if not train_idx or not val_idx:
    raise RuntimeError('Train/val split failed. Adjust val_ratio or dataset size.')

train_tfm = transforms.Compose([
    transforms.RandomResizedCrop(CONFIG['img_size'], scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_tfm = transforms.Compose([
    transforms.Resize(int(CONFIG['img_size'] * 1.14)),
    transforms.CenterCrop(CONFIG['img_size']),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = Subset(FaceAdultDataset(samples, train_tfm, adult_age=CONFIG['adult_age']), train_idx)
val_dataset = Subset(FaceAdultDataset(samples, val_tfm, adult_age=CONFIG['adult_age']), val_idx)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_pin_memory = device.type == 'cuda'

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
    pin_memory=use_pin_memory,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=use_pin_memory,
)

model = ViTBinary(pretrained=CONFIG['pretrained']).to(device)
pos_weight = nonadult_count / max(adult_count, 1)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])

config_to_save = {
    **CONFIG,
    'resolved_data_dir': str(data_root.resolve()),
    'train_size': len(train_idx),
    'val_size': len(val_idx),
    'adult_count': adult_count,
    'nonadult_count': nonadult_count,
    'pos_weight': float(pos_weight),
    'device': str(device),
}

with config_path.open('w', encoding='utf-8') as file:
    json.dump(config_to_save, file, indent=2)

print(f'Dataset: {data_root.resolve()}')
print(f'Total images: {len(samples)} | Train: {len(train_idx)} | Val: {len(val_idx)}')
print(f'Adult(>= {CONFIG["adult_age"]}): {adult_count} | Not adult: {nonadult_count}')
print(f'Device: {device}')
print(f'Using pos_weight={pos_weight:.4f}')


In [ ]:
history = []
best_f1 = -1.0

for epoch in range(1, CONFIG['epochs'] + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, device, criterion)
    val_metrics = evaluate(model, val_loader, device, criterion, threshold=CONFIG['threshold'])
    lr_now = optimizer.param_groups[0]['lr']

    epoch_record = {
        'epoch': epoch,
        'lr': float(lr_now),
        'train_loss': float(train_loss),
        'val_loss': float(val_metrics['loss']),
        'val_acc': float(val_metrics['acc']),
        'val_f1': float(val_metrics['f1']),
        'val_precision': float(val_metrics['precision']),
        'val_recall': float(val_metrics['recall']),
        'tp': int(val_metrics['tp']),
        'tn': int(val_metrics['tn']),
        'fp': int(val_metrics['fp']),
        'fn': int(val_metrics['fn']),
    }
    history.append(epoch_record)

    write_history(history, history_csv_path, history_json_path)

    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'history': history,
        'config': config_to_save,
    }

    if epoch_record['val_f1'] > best_f1:
        best_f1 = epoch_record['val_f1']
        checkpoint['best_val_f1'] = best_f1
        torch.save(checkpoint, best_ckpt_path)

    if CONFIG['save_every_epoch_checkpoint']:
        torch.save(checkpoint, epoch_ckpt_dir / f'epoch_{epoch:03d}.pt')

    print(
        f"Epoch {epoch:03d}/{CONFIG['epochs']} | "
        f"lr={epoch_record['lr']:.6f} | "
        f"train_loss={epoch_record['train_loss']:.4f} | "
        f"val_loss={epoch_record['val_loss']:.4f} | "
        f"val_acc={epoch_record['val_acc']:.4f} | "
        f"val_f1={epoch_record['val_f1']:.4f} | "
        f"val_precision={epoch_record['val_precision']:.4f} | "
        f"val_recall={epoch_record['val_recall']:.4f}"
    )

    scheduler.step()

torch.save(
    {
        'epoch': CONFIG['epochs'],
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'history': history,
        'config': config_to_save,
    },
    last_ckpt_path,
)

print('')
print(f'Best checkpoint: {best_ckpt_path}')
print(f'Last checkpoint: {last_ckpt_path}')
print(f'History CSV: {history_csv_path}')
print(f'History JSON: {history_json_path}')

if pd is not None:
    pd.DataFrame(history)
else:
    history[-5:]
